# Cats and Dogs++ High-Performance Sequential Training Pipeline (Colab Pro)

In [1]:
!pip install -q timm lightning torchmetrics wandb

In [2]:
import zipfile
import shutil
from pathlib import Path

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

root = Path.cwd()
if not (root / "notebooks").exists() and (root.parent / "notebooks").exists():
    root = root.parent

data_target = root / "data"
drive_a2 = Path("/content/drive/MyDrive/DL Data/A2")
search_dirs = [drive_a2, root, Path("/content")]

for d in search_dirs:
    if d.exists():
        for zf in d.glob("*.zip"):
            print(f"Extracting {zf.name} from {d}...")
            with zipfile.ZipFile(zf, "r") as zip_ref:
                zip_ref.extractall(root)
            break
        if (root / "cats-and-dogs").exists() or (root / "data").exists():
            break

possible_folders = [
    root / "cats-and-dogs",
    root / "cats and dogs",
    root / "data",
    Path("/content/cats-and-dogs"),
    Path("/content/cats and dogs"),
    Path("/content/data"),
    drive_a2 / "cats-and-dogs",
    drive_a2 / "cats and dogs",
    drive_a2 / "data",
    drive_a2
]

found_folder = None
for f in possible_folders:
    if f.exists() and (f / "train.csv").exists():
        found_folder = f
        break

if found_folder and found_folder.resolve() != data_target.resolve():
    if data_target.exists():
        shutil.rmtree(data_target)
    shutil.copytree(found_folder, data_target)
    print(f"Prepared data directory at {data_target} from {found_folder.name}")
elif data_target.exists() and (data_target / "train.csv").exists():
    print(f"Data directory ready at {data_target}")

for d in search_dirs:
    if d.exists():
        for f in d.glob("*fold*.csv"):
            dest = root / "folds.csv"
            if f.resolve() != dest.resolve():
                shutil.copy(f, dest)
                print(f"Copied {f.name} to {dest}")
            break
    if (root / "folds.csv").exists():
        break

assert (root / "folds.csv").exists(), "folds.csv not found!"
assert (data_target / "train.csv").exists(), "data/train.csv not found!"
assert (data_target / "train").exists(), "data/train/ directory not found!"
print("All files and folders verified successfully!")

Mounted at /content/drive
Extracting cats-and-dogs.zip from /content/drive/MyDrive/DL Data/A2...
Prepared data directory at /content/data from cats-and-dogs
Copied folds.csv to /content/folds.csv
All files and folders verified successfully!


In [3]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
import timm
import torchmetrics
import lightning.pytorch as pl
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import wandb

torch.set_float32_matmul_precision('high')

In [4]:
class CatsDogsDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.labels = ["Cat", "Dog", "Moris", "Motya", "Biatrix"]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / f"{row['image_id']}.jpg"

        with Image.open(img_path) as img:
            image = img.convert("RGB")
            if self.transform:
                image = self.transform(image)

        target = torch.tensor(row[self.labels].to_numpy(dtype="float32"))
        return image, target


def get_dataloaders(fold_idx=0, batch_size=64, num_workers=4, project_root=None):
    if project_root is None:
        root = Path.cwd()
        if not (root / "data/train.csv").exists():
            root = root.parent
        project_root = root
    else:
        project_root = Path(project_root)

    data_dir = project_root / "data"
    folds_csv_path = project_root / "folds.csv"
    train_csv_path = data_dir / "train.csv"
    img_dir = data_dir / "train"

    df = pd.read_csv(train_csv_path)
    folds = pd.read_csv(folds_csv_path)
    df = df.merge(folds, on="image_id", validate="one_to_one")

    train_df = df[df["fold"] != fold_idx].reset_index(drop=True)
    val_df = df[df["fold"] == fold_idx].reset_index(drop=True)

    labels = ["Cat", "Dog", "Moris", "Motya", "Biatrix"]
    class_counts = train_df[labels].sum().to_numpy()
    total_train_samples = len(train_df)

    pos_weights = torch.tensor(
        [(total_train_samples - count) / max(count, 1) for count in class_counts],
        dtype=torch.float32
    )

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = CatsDogsDataset(train_df, img_dir, train_transform)
    val_dataset = CatsDogsDataset(val_df, img_dir, val_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=bool(num_workers > 0))

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=bool(num_workers > 0))

    return train_loader, val_loader, pos_weights

In [5]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        sq = self.squeeze(x)
        weights = self.excitation(sq)
        return x * weights


class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction, in_channels, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))

        out = avg_out + max_out
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)

        out = torch.cat([avg_out, max_out], dim=1)

        out = self.conv(out)
        return self.sigmoid(out)


class CBAMBlock(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x_out = x * self.channel_attention(x)
        x_out = x_out * self.spatial_attention(x_out)

        return x_out


class PetClassifier(pl.LightningModule):
    def __init__(self, mode="baseline", pos_weights=None, lr=1e-4, optimizer_pair="radam_plateau"):
        super().__init__()
        self.save_hyperparameters(ignore=['pos_weights'])
        self.lr = lr
        self.optimizer_pair = optimizer_pair

        self.backbone = timm.create_model("resnet34", pretrained=True)

        if mode == "baseline":
            self.attention = nn.Identity()
        elif mode == "se":
            self.attention = SEBlock(in_channels=512)
        elif mode == "cbam":
            self.attention = CBAMBlock(in_channels=512)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(512, 5)

        self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

        self.train_f1 = torchmetrics.F1Score(task="multilabel", num_labels=5, average='macro')
        self.val_f1 = torchmetrics.F1Score(task="multilabel", num_labels=5, average='macro')
        self.val_f1_per_class = torchmetrics.F1Score(task="multilabel", num_labels=5, average='none')

    def forward(self, x):
        features = self.backbone.forward_features(x)
        recalibrated_features = self.attention(features)
        pooled = self.global_pool(recalibrated_features)
        flat = torch.flatten(pooled, 1)
        logits = self.head(flat)
        return logits

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)

        self.train_f1(logits, y)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_macro_f1', self.train_f1, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)

        self.val_f1(logits, y)
        self.val_f1_per_class(logits, y)

        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self.log('val_macro_f1', self.val_f1, on_epoch=True, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        per_class_f1 = self.val_f1_per_class.compute()
        classes = ["Cat", "Dog", "Moris", "Motya", "Biatrix"]

        for i, class_name in enumerate(classes):
            self.log(f'val_f1_{class_name}', per_class_f1[i])

        self.val_f1_per_class.reset()

    def on_before_optimizer_step(self, optimizer):
        if self.global_step % 10 == 0:
            block_norms = {}
            for name, param in self.named_parameters():
                if param.grad is not None:
                    norm = param.grad.detach().norm(2).item()
                    for block in ["head", "attention", "layer4", "layer3", "layer2", "layer1"]:
                        if block in name:
                            block_norms.setdefault(block, []).append(norm)
                            break
            for block, norms in block_norms.items():
                self.log(f"grad_flow/{block}", sum(norms) / len(norms), on_step=True)

    def configure_optimizers(self):
        if self.optimizer_pair == "radam_plateau":
            optimizer = torch.optim.RAdam(self.parameters(), lr=self.lr, weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-6
            )
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "monitor": "val_macro_f1"
                }
            }
        elif self.optimizer_pair == "sgd_onecycle":
            optimizer = torch.optim.SGD(
                self.parameters(), lr=self.lr, momentum=0.9, nesterov=True, weight_decay=1e-4
            )
            scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=self.lr,
                total_steps=self.trainer.estimated_stepping_batches
            )
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "step"
                }
            }
        else:
            optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
            return {"optimizer": optimizer, "lr_scheduler": scheduler}


def setup_trainer(mode, optimizer_pair="radam_plateau", max_epochs=10):
    wandb_logger = WandbLogger(
        project="AI301_Assignment2",
        name=f"resnet34_{mode}_{optimizer_pair}_fold0",
        log_model="all"
    )

    checkpoint_callback = ModelCheckpoint(
        dirpath="checkpoints/",
        monitor="val_macro_f1",
        mode="max",
        save_top_k=1,
        filename=f"best-resnet34-{mode}-{optimizer_pair}-{{epoch:02d}}-{{val_macro_f1:.3f}}"
    )

    early_stop_callback = EarlyStopping(
        monitor="val_macro_f1",
        patience=3,
        mode="max",
        verbose=True
    )

    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        prec = "bf16-mixed"
    elif torch.cuda.is_available():
        prec = "16-mixed"
    else:
        prec = "32-true"

    trainer = pl.Trainer(
        logger=wandb_logger,
        callbacks=[checkpoint_callback, early_stop_callback],
        max_epochs=max_epochs,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        precision=prec,
        log_every_n_steps=10,
        fast_dev_run=False
    )
    return trainer

In [6]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]

        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_full_backward_hook(backward_hook))

    def generate(self, input_tensor, target_class_idx):
        self.model.zero_grad()
        logits = self.model(input_tensor)
        score = logits[:, target_class_idx].sum()
        score.backward(retain_graph=True)

        gradients = self.gradients
        activations = self.activations

        weights = torch.mean(gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * activations, dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam = F.interpolate(cam, size=(input_tensor.shape[2], input_tensor.shape[3]), mode="bilinear", align_corners=False)
        return cam.squeeze().detach().cpu().numpy(), logits.squeeze().detach().cpu()

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()


def overlay_cam(img_np, cam, alpha=0.5):
    cmap = plt.get_cmap("jet")
    heatmap = cmap(cam)[:, :, :3]
    overlay = (1 - alpha) * img_np + alpha * heatmap
    return np.clip(overlay, 0, 1), heatmap


def process_and_save(grad_cam, image_tensor, original_img, class_idx, class_name, output_path, device):
    input_tensor = image_tensor.unsqueeze(0).to(device)
    cam, logits = grad_cam.generate(input_tensor, class_idx)
    prob = torch.sigmoid(logits)[class_idx].item()
    logit_val = logits[class_idx].item()

    img_resized = original_img.resize((224, 224))
    img_np = np.array(img_resized).astype(np.float32) / 255.0
    overlay, heatmap = overlay_cam(img_np, cam)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_np)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(heatmap)
    axes[1].set_title("Grad-CAM Heatmap")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title(f"{class_name} (Prob: {prob:.3f}, Logit: {logit_val:.2f})")
    axes[2].axis("off")

    plt.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close(fig)
    print(f"Saved: {output_path}")


def run_gradcam(checkpoint_path, mode="baseline", output_dir="reports/gradcam", image_path=None, num_samples=5, target_class=None, run_name=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    labels = ["Cat", "Dog", "Moris", "Motya", "Biatrix"]

    model = PetClassifier.load_from_checkpoint(checkpoint_path, mode=mode, strict=False)
    model.eval()
    model.to(device)

    target_layer = model.attention
    grad_cam = GradCAM(model, target_layer)

    sub_dir = run_name if run_name else mode
    out_dir = Path(output_dir) / sub_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    if image_path:
        img_path = Path(image_path)
        with Image.open(img_path) as img:
            original_img = img.convert("RGB")
        image_tensor = transform(original_img)

        with torch.no_grad():
            preds = model(image_tensor.unsqueeze(0).to(device))
            top_class = target_class if target_class else labels[preds.argmax().item()]

        target_idx = labels.index(top_class)
        save_path = out_dir / f"{img_path.stem}_{top_class}.png"
        process_and_save(grad_cam, image_tensor, original_img, target_idx, top_class, save_path, device)
    else:
        _, val_loader, _ = get_dataloaders(fold_idx=0, batch_size=1, num_workers=0)
        val_df = val_loader.dataset.df

        for idx in range(min(num_samples, len(val_df))):
            row = val_df.iloc[idx]
            image_id = row["image_id"]
            img_path = val_loader.dataset.img_dir / f"{image_id}.jpg"
            if not img_path.exists():
                continue

            with Image.open(img_path) as img:
                original_img = img.convert("RGB")
            image_tensor = transform(original_img)

            if target_class:
                target_classes = [target_class]
            else:
                active_classes = [lbl for lbl in labels if row[lbl] == 1]
                target_classes = active_classes if active_classes else [labels[0]]

            for cls_name in target_classes:
                cls_idx = labels.index(cls_name)
                save_path = out_dir / f"{image_id}_{cls_name}.png"
                process_and_save(grad_cam, image_tensor, original_img, cls_idx, cls_name, save_path, device)

    grad_cam.remove_hooks()


In [7]:
def find_optimal_thresholds(ckpt_path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    _, val_loader, _ = get_dataloaders(fold_idx=0, batch_size=64, num_workers=4)

    print(f"Loading checkpoint: {ckpt_path}")
    model = PetClassifier.load_from_checkpoint(ckpt_path, strict=False)
    model.eval()
    model.to(device)

    all_probs = []
    all_targets = []

    print("Running validation forward passes...")
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)

    labels = ["Cat", "Dog", "Moris", "Motya", "Biatrix"]
    best_thresholds = []

    thresholds = np.linspace(0.01, 0.99, 99)

    print("Optimizing Thresholds...")
    for i, class_name in enumerate(labels):
        class_probs = all_probs[:, i]
        class_targets = all_targets[:, i]

        best_f1 = 0
        best_t = 0.5

        for t in thresholds:
            preds = (class_probs >= t).astype(int)

            score = f1_score(class_targets, preds)

            if score > best_f1:
                best_f1 = score
                best_t = t

        best_thresholds.append(best_t)
        print(f"[{class_name}] Best Threshold: {best_t:.2f} | Validation F1 Score: {best_f1:.4f}")

    print(f"Final Threshold Array:")
    print(f"{best_thresholds}")

    overall_preds = (all_probs >= np.array(best_thresholds)).astype(int)
    macro_f1 = f1_score(all_targets, overall_preds, average='macro')
    print(f"OPTIMIZED GLOBAL MACRO-F1: {macro_f1:.4f}")
    return best_thresholds

In [8]:
class CatsDogsTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / f"{row['image_id']}.jpg"

        with Image.open(img_path) as img:
            image = img.convert("RGB")
            if self.transform:
                image = self.transform(image)

        return image, row['image_id']


def run_inference(ckpt_path, test_csv_path, img_dir, output_csv, thresholds=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running inference on: {device}")

    print(f"Loading checkpoint: {ckpt_path}")
    model = PetClassifier.load_from_checkpoint(ckpt_path, strict=False)
    model.eval()
    model.to(device)

    df_test = pd.read_csv(test_csv_path)

    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_dataset = CatsDogsTestDataset(df_test, img_dir, test_transform)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

    image_ids = []
    all_predictions = []

    print("Running forward passes...")
    if thresholds is None:
        optimal_thresholds = torch.tensor([0.75, 0.29, 0.89, 0.99, 0.97], device=device)
    else:
        optimal_thresholds = torch.tensor(thresholds, device=device)

    with torch.no_grad():
        for images, batch_img_ids in test_loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.sigmoid(logits)
            probs_interpreted = (probs >= optimal_thresholds).int()

            image_ids.extend(batch_img_ids)
            all_predictions.extend(probs_interpreted.cpu().numpy())

    print("Formatting submission...")
    labels = ["Cat", "Dog", "Moris", "Motya", "Biatrix"]

    submission_df = pd.DataFrame(all_predictions, columns=labels)
    submission_df.insert(0, 'image_id', image_ids)

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    submission_df.to_csv(output_csv, index=False)
    print(f"Submission saved to {output_csv}")

### Step 1: Sequential Optimizer Comparison on Baseline ResNet-34

In [9]:
wandb.login()

train_loader, val_loader, pos_weights = get_dataloaders(fold_idx=0, batch_size=64, num_workers=4)

baseline_results = {}
pairs_to_evaluate = [
    ("radam_plateau", 1e-4),
    ("sgd_onecycle", 1e-2)
]

for pair_name, lr_val in pairs_to_evaluate:
    print(f"{'='*25}Starting Training: baseline with {pair_name}{'='*25}")
    model = PetClassifier(mode="baseline", pos_weights=pos_weights, lr=lr_val, optimizer_pair=pair_name)
    trainer = setup_trainer(mode="baseline", optimizer_pair=pair_name, max_epochs=10)
    trainer.fit(model, train_loader, val_loader)

    best_score = trainer.checkpoint_callback.best_model_score.item()
    best_path = trainer.checkpoint_callback.best_model_path
    baseline_results[pair_name] = {"score": best_score, "path": best_path}

    wandb.finish()
    del model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

winning_pair = max(baseline_results, key=lambda k: baseline_results[k]["score"])
print(f"{'='*30}BASELINE COMPARISON RESULTS:{'='*30}")
for k, v in baseline_results.items():
    print(f"Pair: {k:15s} | Best Val Macro-F1: {v['score']:.4f} | Checkpoint: {v['path']}")
print(f"WINNING PAIR: {winning_pair} (Score: {baseline_results[winning_pair]['score']:.4f})")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 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


wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


... (17 outputs left out)

Output()

INFO: Metric val_macro_f1 improved. New best score: 0.417
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_macro_f1 improved. New best score: 0.417
INFO: Metric val_macro_f1 improved by 0.022 >= min_delta = 0.0. New best score: 0.439
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_macro_f1 improved by 0.022 >= min_delta = 0.0. New best score: 0.439
INFO: Metric val_macro_f1 improved by 0.092 >= min_delta = 0.0. New best score: 0.531
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_macro_f1 improved by 0.092 >= min_delta = 0.0. New best score: 0.531
INFO: Metric val_macro_f1 improved by 0.053 >= min_delta = 0.0. New best score: 0.583
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_macro_f1 improved by 0.053 >= min_delta = 0.0. New best score: 0.583
INFO: Metric val_macro_f1 improved by 0.038 >= min_delta = 0.0. New best score: 0.622
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_macro_f1 improved by 0.038 >= min_delta = 0.0. New best

epoch,▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇█████
grad_flow/head,▃▃▄▆▂▆▃▅█▂▂▃▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁
grad_flow/layer1,▁▂▂▁▃▁▁▂▂▃▁▁▄▁▄▁▁▁█▁▁▇▁▁▁▁▁▁▂▁▄▄▁▁▁▂▁▁▁▁
grad_flow/layer2,▂▂▁▂▁▃▁█▃▃▁▁▁▁▁▁▁▁▁▁▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁
grad_flow/layer3,▂▂▂▁▁▃▂▃▅█▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁
grad_flow/layer4,▄▁▃▂▁▅▄▁▅▁▁▂█▁▁▁▁█▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss_epoch,█▅▃▂▂▂▁▁▁▁
train_loss_step,▄█▄▄▅▃▃▂▄▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_macro_f1,▁▃▄▅▆▅▇███
trainer/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
+7,...


==============================BASELINE COMPARISON RESULTS:==============================
Pair: radam_plateau   | Best Val Macro-F1: 0.7846 | Checkpoint: /content/checkpoints/best-resnet34-baseline-radam_plateau-epoch=09-val_macro_f1=0.785.ckpt
Pair: sgd_onecycle    | Best Val Macro-F1: 0.6781 | Checkpoint: /content/checkpoints/best-resnet34-baseline-sgd_onecycle-epoch=09-val_macro_f1=0.678.ckpt
WINNING PAIR: radam_plateau (Score: 0.7846)


### Step 2: Choose Optimizer Pair for Attention Architectures (SE & CBAM)

In [10]:
# Possible parameters for SELECTED_PAIR:
# "auto"           - Automatically selects the winning pair from Step 1 above (Recommended)
# "pair1"          - RAdam with ReduceLROnPlateau
# "radam_plateau"  - RAdam with ReduceLROnPlateau
# "pair2"          - Nesterov SGD with OneCycleLR
# "sgd_onecycle"   - Nesterov SGD with OneCycleLR
# "adamw_cosine"   - Standard baseline AdamW with Cosine Annealing

SELECTED_PAIR = "auto"

### Step 3: Sequential Training of SE and CBAM with Chosen Pair

In [12]:
if SELECTED_PAIR == "auto":
    chosen_pair = winning_pair
elif SELECTED_PAIR in ["pair1", "radam_plateau"]:
    chosen_pair = "radam_plateau"
elif SELECTED_PAIR in ["pair2", "sgd_onecycle"]:
    chosen_pair = "sgd_onecycle"
else:
    chosen_pair = SELECTED_PAIR

chosen_lr = 1e-2 if chosen_pair == "sgd_onecycle" else 1e-4
print(f"Selected Optimizer Pair: {chosen_pair} (Learning Rate: {chosen_lr})")

attention_results = {}
attention_architectures = ["se", "cbam"]

for arch_mode in attention_architectures:
    print(f"{'='*25}Starting Training: {arch_mode} with {chosen_pair}{'='*25}")
    model = PetClassifier(mode=arch_mode, pos_weights=pos_weights, lr=chosen_lr, optimizer_pair=chosen_pair)
    trainer = setup_trainer(mode=arch_mode, optimizer_pair=chosen_pair, max_epochs=10)
    trainer.fit(model, train_loader, val_loader)

    best_score = trainer.checkpoint_callback.best_model_score.item()
    best_path = trainer.checkpoint_callback.best_model_path
    attention_results[arch_mode] = {"score": best_score, "path": best_path}

    wandb.finish()
    del model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"{'='*30}ALL MODELS TRAINED SUCCESSFULLY:{'='*30}")
for arch, res in attention_results.items():
    print(f"Architecture: {arch:6s} | Score: {res['score']:.4f} | Path: {res['path']}")

Selected Optimizer Pair: radam_plateau (Learning Rate: 0.0001)
=========================Starting Training: se with radam_plateau=========================


INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/usr/local/lib/python3.13/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone         │ ResNet            │ 21.8 M │ train │     0 │
│ 1 │ attention        │ SEBlock           │ 33.3 K │ train │     0 │
│ 2 │ global_pool      │ AdaptiveAvgPool2d │      0 │ train │     0 │
│ 3 │ head             │ Linear            │  2.6 K │ train │     0 │
│ 4 │ criterion        │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 5 │ train_f1         │ MultilabelF1Score │      0 │ train │     0 │
│ 6 │ val_f1           │ MultilabelF1Score │      0 │ train │     0 │
│ 7 │ val_f1_per_class │ MultilabelF1Score │      0 │ train │     0 │
└───┴──────────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 21.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 21.8 M                                                                                               
Total estimated model params size (MB): 87.334                                                                     
Modules in train mode: 179                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

... (9 outputs left out)

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: Metric val_macro_f1 improved. New best score: 0.233
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_macro_f1 improved. New best score: 0.233
INFO: Metric val_macro_f1 improved by 0.027 >= min_delta = 0.0. New best score: 0.259
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_macro_f1 improved by 0.027 >= min_delta = 0.0. New best score: 0.259
INFO: Metric val_macro_f1 improved by 0.051 >= min_delta = 0.0. New best score: 0.310
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_macro_f1 improved by 0.051 >= min_delta = 0.0. New best score: 0.310
INFO: Metric val_macro_f1 improved by 0.074 >= min_delta = 0.0. New best score: 0.384
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_macro_f1 improved by 0.074 >= min_delta = 0.0. New best score: 0.384
INFO: Metric val_macro_f1 improved by 0.108 >= min_delta = 0.0. New best score: 0.493
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_macro_f1 improved by 0.108 >= min_delta = 0.0. New best

epoch,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇██
grad_flow/attention,▁▁▂▂▁▁▁▄▂▁▁▃▂█▂█▂▂▆▂▆▅▄▃▃▂▃▃▂▂▁▂▁▁▁▁▁▁▁▁
grad_flow/head,▄▂▄▂▂▃▃▂▂▂▂▂▅▂▅▅▂█▃▄▂▄▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
grad_flow/layer1,▂▄▂▃▁▁▁▄▁▅▆▁▁▆▁▂▃▂▂▂█▂▄▃▂▁▂▁▂▂▁▂▁▃▄▁▂▁▂▁
grad_flow/layer2,▁▁▃▃▃▃▅▁▆▁▁██▂▂▂▂▂▃▂▃▄▃▂▂▃▁▂▂▂▂▂▂▂▂▂▂▁▂▂
grad_flow/layer3,▁▁▂▂▂▁▁▁▁▃▁▁█▁▅▁▃▂▁▁▂▁▁▃▂▂▂▅▁▂▁▂▂▁▁▁▁▁▁▂
grad_flow/layer4,▁▁▂▁▂▁▁▁▁▂▁▁▄▁▃▂▄▂▂▂▃▂▂▂▃▂▂▂▁▂▁▁▁█▁▂▁▁▁▁
train_loss_epoch,██▇▆▄▂▂▁▁▁
train_loss_step,▃▃▃▃▃▃▃▅▅▃▅█▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train_macro_f1,▁▂▂▃▄▅▇▇▇█
+8,...


==============================ALL MODELS TRAINED SUCCESSFULLY:==============================
Architecture: se     | Score: 0.7914 | Path: /content/checkpoints/best-resnet34-se-radam_plateau-epoch=08-val_macro_f1=0.791.ckpt
Architecture: cbam   | Score: 0.7662 | Path: /content/checkpoints/best-resnet34-cbam-radam_plateau-epoch=09-val_macro_f1=0.766.ckpt


### Step 4: Automated Sequential Evaluation (Grad-CAM & Threshold Optimization)

In [ ]:
if "baseline_results" not in locals() or "attention_results" not in locals():
    print("In-memory training results not found (training steps 1-3 were skipped).")
    print("Using the checkpoint loader cell below instead.")
else:
    all_checkpoints = {"baseline": baseline_results[winning_pair]["path"]}
    all_checkpoints.update({k: v["path"] for k, v in attention_results.items()})

    for arch_mode, ckpt_file in all_checkpoints.items():
        print(f"Generating Grad-CAM for {arch_mode}...")
        run_gradcam(checkpoint_path=ckpt_file, mode=arch_mode, output_dir="reports/gradcam", num_samples=5)

        print(f"Optimizing Thresholds for {arch_mode}...")
        find_optimal_thresholds(ckpt_file)


### Step 4 (Alternative): Load Checkpoints from Google Drive & Evaluate
If you downloaded the checkpoints from WandB into Google Drive (), run this cell to evaluate without re-training Steps 1-3.

In [ ]:
from pathlib import Path

# Search for checkpoints in Google Drive or local paths
search_dirs = [
    Path("/content/drive/MyDrive/DL Data/checkpoints"),
    Path("/content/drive/MyDrive/DL Data/checkpoints/"),
    Path("/content/drive/Shareddrives/DL Data/checkpoints"),
    Path("DL Data/checkpoints"),
    Path("/content/checkpoints"),
    Path("checkpoints")
]

ckpt_dir = None
for candidate in search_dirs:
    if candidate.exists() and any(candidate.glob("*.ckpt")):
        ckpt_dir = candidate
        break

if ckpt_dir is None:
    ckpt_dir = Path("/content/drive/MyDrive/DL Data/checkpoints")

print(f"Searching for checkpoints in: {ckpt_dir}")

def resolve_ckpt(directory, pattern):
    if not directory.exists():
        return None
    if (directory / pattern).is_file():
        return str(directory / pattern)
    matches = sorted(list(directory.glob(f"*{pattern}*.ckpt")), key=lambda p: p.stat().st_mtime, reverse=True)
    return str(matches[0]) if matches else None

# The 4 models from the experimental runs:
target_runs = [
    ("cbam_radam_plateau", "cbam", ["cbam*radam", "cbam"]),
    ("se_radam_plateau", "se", ["se*radam", "se"]),
    ("baseline_sgd_onecycle", "baseline", ["baseline*sgd", "sgd_onecycle"]),
    ("baseline_radam_plateau", "baseline", ["baseline*radam", "radam_plateau"])
]

all_checkpoints = {}
for run_name, mode, patterns in target_runs:
    ckpt_path = None
    for pat in patterns:
        ckpt_path = resolve_ckpt(ckpt_dir, pat)
        if ckpt_path:
            break
    if ckpt_path:
        all_checkpoints[run_name] = {"path": ckpt_path, "mode": mode}
        print(f"  [{run_name:24s}] Loaded: {ckpt_path}")
    else:
        print(f"  [{run_name:24s}] Warning: No matching checkpoint found in {ckpt_dir}")

assert len(all_checkpoints) > 0, f"No .ckpt files found in {ckpt_dir}! Please check that files are present."

# Automated Sequential Evaluation (Grad-CAM & Threshold Optimization)
optimized_thresholds = {}
for run_name, info in all_checkpoints.items():
    ckpt_file = info["path"]
    arch_mode = info["mode"]
    sep = "=" * 25
    print(f"\n{sep} Evaluating {run_name} ({arch_mode}) {sep}")
    print(f"Generating Grad-CAM for {run_name}...")
    run_gradcam(
        checkpoint_path=ckpt_file,
        mode=arch_mode,
        output_dir="reports/gradcam",
        num_samples=5,
        run_name=run_name
    )

    print(f"Optimizing Thresholds for {run_name}...")
    best_t = find_optimal_thresholds(ckpt_file)
    optimized_thresholds[run_name] = best_t


### Step 5: Test Inference & Submission Generation
Generate Kaggle submission file using your best model (SE) and optimal thresholds.

In [ ]:
# Select the best model checkpoint (SE was highest with 0.7914 validation F1)
best_model_mode = "se"
if best_model_mode in all_checkpoints:
    best_ckpt = all_checkpoints[best_model_mode]
    print(f"Generating test predictions using: {best_ckpt}")

    # Optimal thresholds (update with the values printed by find_optimal_thresholds above)
    optimal_thresholds = [0.75, 0.29, 0.89, 0.99, 0.97]

    test_csv = data_target / "test.csv" if "data_target" in locals() and (data_target / "test.csv").exists() else "data/test.csv"
    test_img = data_target / "test" if "data_target" in locals() and (data_target / "test").exists() else "data/test"

    run_inference(
        ckpt_path=best_ckpt,
        test_csv_path=test_csv,
        img_dir=test_img,
        output_csv=f"submissions/submission_{best_model_mode}_optimized.csv",
        thresholds=optimal_thresholds
    )
else:
    print(f"'{best_model_mode}' checkpoint not found in all_checkpoints.")
